# Benchmark: Remote one-sample-at-a-time (fsspec)


In [ ]:
import io
import json
import os
import time
from datetime import datetime, timezone
from pathlib import Path

import torch
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

try:
    import fsspec
except ImportError as e:
    raise ImportError('Missing dependency: fsspec (pip install fsspec s3fs)') from e

try:
    from PIL import Image
except ImportError as e:
    raise ImportError('Missing dependency: Pillow (pip install pillow)') from e

print('torch:', torch.__version__)


## Configuration


In [ ]:
S3_BUCKET = os.environ.get('S3_BUCKET', '')
S3_PREFIX = os.environ.get('S3_PREFIX', 'Food-11')
SPLIT = os.environ.get('FOOD11_SPLIT', 'evaluation')
S3_ENDPOINT_URL = os.environ.get('S3_ENDPOINT_URL', '')

BATCH_SIZE = int(os.environ.get('BATCH_SIZE', '64'))
NUM_WORKERS_LIST = [0, 2, 4, 8]

WARMUP_BATCHES = 10
MEASURE_BATCHES = 50
MAX_SAMPLES = int(os.environ.get('MAX_SAMPLES', '0'))  # 0 means no cap

if not S3_BUCKET:
    raise ValueError('S3_BUCKET env var is required')

print('S3_BUCKET:', S3_BUCKET)
print('S3_PREFIX:', S3_PREFIX)
print('SPLIT:', SPLIT)
print('S3_ENDPOINT_URL:', S3_ENDPOINT_URL if S3_ENDPOINT_URL else '(default)')
print('BATCH_SIZE:', BATCH_SIZE)
print('NUM_WORKERS_LIST:', NUM_WORKERS_LIST)
print('WARMUP_BATCHES:', WARMUP_BATCHES)
print('MEASURE_BATCHES:', MEASURE_BATCHES)
print('MAX_SAMPLES:', MAX_SAMPLES)


## Dataset (remote index + per-sample reads)


In [ ]:
# Build an index of remote objects (listing happens once; not part of throughput timing).
fs_kwargs = {}
if S3_ENDPOINT_URL:
    fs_kwargs['client_kwargs'] = {'endpoint_url': S3_ENDPOINT_URL}
fs = fsspec.filesystem('s3', **fs_kwargs)

base = f"{S3_BUCKET}/{S3_PREFIX}/{SPLIT}"
pattern = f"{base}/class_*/*"
paths = fs.glob(pattern)
paths = [p for p in paths if not p.endswith('/') ]
paths.sort()

samples = []
for p in paths:
    # p like: bucket/prefix/split/class_00/0_123.jpg
    parts = p.split('/')
    try:
        cls = next(seg for seg in parts if seg.startswith('class_'))
        label = int(cls.split('_')[1])
    except Exception:
        continue
    samples.append({'path': p, 'label': label})

if MAX_SAMPLES and MAX_SAMPLES < len(samples):
    samples = samples[:MAX_SAMPLES]

print('indexed_samples:', len(samples))

transform = transforms.Compose([transforms.ToTensor()])

class RemoteImageDataset(Dataset):
    def __init__(self, samples, fs_kwargs, transform=None):
        self.samples = samples
        self.fs_kwargs = fs_kwargs
        self.transform = transform
        self._fs = None

    def _get_fs(self):
        if self._fs is None:
            self._fs = fsspec.filesystem('s3', **self.fs_kwargs)
        return self._fs

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        fs = self._get_fs()
        with fs.open(s['path'], 'rb') as f:
            b = f.read()
        img = Image.open(io.BytesIO(b)).convert('RGB')
        if self.transform is not None:
            img = self.transform(img)
        return img, int(s['label'])

dataset = RemoteImageDataset(samples=samples, fs_kwargs=fs_kwargs, transform=transform)


## Run benchmark


In [ ]:
results = []

for num_workers in NUM_WORKERS_LIST:
    loader_kwargs = {
        'batch_size': BATCH_SIZE,
        'shuffle': False,
        'num_workers': num_workers,
        'pin_memory': False,
        'drop_last': False,
    }
    if num_workers > 0:
        loader_kwargs['prefetch_factor'] = 2
        loader_kwargs['persistent_workers'] = True

    loader = DataLoader(dataset, **loader_kwargs)
    it = iter(loader)

    t0 = time.perf_counter()
    x0, y0 = next(it)
    t1 = time.perf_counter()
    time_to_first_batch_s = t1 - t0

    for _ in range(WARMUP_BATCHES):
        try:
            _ = next(it)
        except StopIteration:
            break

    num_batches = 0
    num_items = 0
    t_start = time.perf_counter()
    for _ in range(MEASURE_BATCHES):
        try:
            x, y = next(it)
        except StopIteration:
            break
        num_batches += 1
        num_items += int(y.shape[0])
    t_end = time.perf_counter()

    wall_s = t_end - t_start
    imgs_per_s = (num_items / wall_s) if wall_s > 0 else float('nan')
    batches_per_s = (num_batches / wall_s) if wall_s > 0 else float('nan')
    avg_batch_s = (wall_s / num_batches) if num_batches > 0 else None

    results.append({
        'num_workers': num_workers,
        'batch_size': BATCH_SIZE,
        'time_to_first_batch_s': time_to_first_batch_s,
        'measured_batches': num_batches,
        'measured_items': num_items,
        'wall_s': wall_s,
        'imgs_per_s': imgs_per_s,
        'batches_per_s': batches_per_s,
        'avg_batch_s': avg_batch_s,
    })

results


## Print results


In [ ]:
print('split:', SPLIT)
print('batch_size:', BATCH_SIZE)
print('warmup_batches:', WARMUP_BATCHES, 'measure_batches:', MEASURE_BATCHES)
print()

for r in results:
    avg_batch_s = r['avg_batch_s']
    avg_batch_s_str = 'nan' if avg_batch_s is None else f"{avg_batch_s:.4f}"
    print(
        'workers=', r['num_workers'],
        'imgs/s=', f"{r['imgs_per_s']:.2f}",
        'batches/s=', f"{r['batches_per_s']:.2f}",
        'first_batch_s=', f"{r['time_to_first_batch_s']:.3f}",
        'avg_batch_s=', avg_batch_s_str,
    )

imgs_per_s_values = [r['imgs_per_s'] for r in results if r['measured_batches'] > 0]
if imgs_per_s_values:
    best = max(results, key=lambda x: x['imgs_per_s'])
    mean_imgs_per_s = sum(imgs_per_s_values) / len(imgs_per_s_values)
    print()
    print('aggregate_mean_imgs_per_s:', round(mean_imgs_per_s, 2))
    print('aggregate_best_imgs_per_s:', round(best['imgs_per_s'], 2), 'at num_workers=', best['num_workers'])
else:
    print('No measured batches; check dataset and settings.')


## Save results


In [ ]:
out_dir = Path('results')
out_dir.mkdir(parents=True, exist_ok=True)

stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
out_path = out_dir / f'remote_one_sample_{stamp}.json'
payload = {
    'benchmark': 'remote_one_sample',
    'timestamp_utc': stamp,
    's3_bucket': S3_BUCKET,
    's3_prefix': S3_PREFIX,
    'split': SPLIT,
    'batch_size': BATCH_SIZE,
    'warmup_batches': WARMUP_BATCHES,
    'measure_batches': MEASURE_BATCHES,
    'max_samples': MAX_SAMPLES,
    'results': results,
}
out_path.write_text(json.dumps(payload, indent=2))
print('Wrote:', out_path)
